In [20]:
import kagglehub
from pycocotools.coco import COCO
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

path = kagglehub.dataset_download(
    "awsaf49/coco-2017-dataset",
    "coco2017/annotations/instances_train2017.json"
)

# Initialize COCO API
coco = COCO(path)

loading annotations into memory...
Done (t=14.66s)
creating index...
index created!


In [21]:
# Get category IDs to fetch subsets
category_ids = coco.getCatIds(catNms=['person', 'car', 'bicycle', 'truck', 'bus'])

# Get image Ids for these categories
# image_ids = coco.getImgIds(catIds=category_ids)
# Get all image IDs that contain any of our target classes
image_ids = set()
for category_id in category_ids:
    class_ids = coco.getImgIds(catIds=[category_id])
    image_ids.update(class_ids)
image_ids = list(image_ids)
print(len(image_ids))

# Get annotation Ids for these categories
ann_ids = coco.getAnnIds(imgIds=image_ids, catIds=category_ids)
print(len(ann_ids))

# Get categories
categories = coco.loadCats(category_ids)

69655
329487


In [22]:
# Load data into dataframes
images_df = pd.DataFrame(coco.loadImgs(image_ids)).rename(columns={'id': 'image_id'})
ann_df = pd.DataFrame(coco.loadAnns(ann_ids))
category_df = pd.DataFrame(categories).rename(columns={'id': 'category_id'})

In [23]:
# Link annotations to category names
merged = pd.merge(ann_df, category_df[['category_id', 'name', 'supercategory']], on='category_id')
# Link annotations to image metadata
final = pd.merge(merged, images_df, on='image_id')

final.head()

,segmentation,area,iscrowd,image_id,bbox,category_id,id,name,supercategory,license,file_name,coco_url,height,width,date_captured,flickr_url
0,"[[453.0, 292.1, 457.0, 253.1, 439.0, 245.1, 43...",21258.00000,0,262145,"[387.0, 71.1, 145.0, 322.0]",1,1218400,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...
1,"[[505.24, 98.89, 505.4, 96.89, 505.7, 95.04, 5...",74.23865,0,262145,"[494.31, 91.04, 11.7, 16.16]",1,1220136,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...
2,"[[282.37, 174.17, 282.91, 161.84, 282.91, 145....",2053.16515,0,262145,"[282.37, 86.79, 32.16, 97.02]",1,1246711,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...
3,"[[526.17, 131.72, 530.84, 141.64, 536.68, 142....",9110.87005,0,262145,"[526.17, 44.14, 90.5, 188.0]",1,1249960,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...
4,"[[508.34, 166.53, 513.54, 161.32, 516.35, 153....",1962.25295,0,262145,"[491.51, 81.19, 40.46, 85.34]",1,1262984,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...


In [24]:
final.shape

(329487, 16)

In [25]:
# Remove null rows
cleaned_df = final.dropna(how='any')
cleaned_df.shape

(329487, 16)

In [26]:
# Map COCO-original indices to 0-index for cleaner processing
map_ids = {old_id: index for index, old_id in enumerate(category_ids)}

# Create new column in dataframe to include 0-based indexing
cleaned_df['image_index'] = cleaned_df['category_id'].map(map_ids)

In [27]:
cleaned_df.head()

,segmentation,area,iscrowd,image_id,bbox,category_id,id,name,supercategory,license,file_name,coco_url,height,width,date_captured,flickr_url,image_index
0,"[[453.0, 292.1, 457.0, 253.1, 439.0, 245.1, 43...",21258.00000,0,262145,"[387.0, 71.1, 145.0, 322.0]",1,1218400,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...,0
1,"[[505.24, 98.89, 505.4, 96.89, 505.7, 95.04, 5...",74.23865,0,262145,"[494.31, 91.04, 11.7, 16.16]",1,1220136,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...,0
2,"[[282.37, 174.17, 282.91, 161.84, 282.91, 145....",2053.16515,0,262145,"[282.37, 86.79, 32.16, 97.02]",1,1246711,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...,0
3,"[[526.17, 131.72, 530.84, 141.64, 536.68, 142....",9110.87005,0,262145,"[526.17, 44.14, 90.5, 188.0]",1,1249960,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...,0
4,"[[508.34, 166.53, 513.54, 161.32, 516.35, 153....",1962.25295,0,262145,"[491.51, 81.19, 40.46, 85.34]",1,1262984,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...,0


In [28]:
# Rearrange to make the index column to the 0th index

cols = cleaned_df.columns.tolist()
# Remove index column from current index and insert at 0th index
cols.insert(0, cols.pop(cols.index('image_index')))
# Reassign the dataframe with the new column order
cleaned_df = cleaned_df[cols]

In [29]:
cleaned_df.head()

,image_index,segmentation,area,iscrowd,image_id,bbox,category_id,id,name,supercategory,license,file_name,coco_url,height,width,date_captured,flickr_url
0,0,"[[453.0, 292.1, 457.0, 253.1, 439.0, 245.1, 43...",21258.00000,0,262145,"[387.0, 71.1, 145.0, 322.0]",1,1218400,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...
1,0,"[[505.24, 98.89, 505.4, 96.89, 505.7, 95.04, 5...",74.23865,0,262145,"[494.31, 91.04, 11.7, 16.16]",1,1220136,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...
2,0,"[[282.37, 174.17, 282.91, 161.84, 282.91, 145....",2053.16515,0,262145,"[282.37, 86.79, 32.16, 97.02]",1,1246711,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...
3,0,"[[526.17, 131.72, 530.84, 141.64, 536.68, 142....",9110.87005,0,262145,"[526.17, 44.14, 90.5, 188.0]",1,1249960,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...
4,0,"[[508.34, 166.53, 513.54, 161.32, 516.35, 153....",1962.25295,0,262145,"[491.51, 81.19, 40.46, 85.34]",1,1262984,person,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...


In [30]:
# Delete columns
delete_columns = ['iscrowd']
cleaned_df.drop(columns=delete_columns, inplace=True)

In [31]:
# One-hot Encoding
categorical_features = ['name']
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)], remainder='passthrough', verbose_feature_names_out=False)

# Fit and transform the data
encoded_df = ct.fit_transform(cleaned_df)

# Convert back to dataframe with column names
feature_names = ct.get_feature_names_out()
cleaned_df = pd.DataFrame(encoded_df, columns=feature_names)


cleaned_df.head()


,name_bicycle,name_bus,name_car,name_person,name_truck,image_index,segmentation,area,image_id,bbox,category_id,id,supercategory,license,file_name,coco_url,height,width,date_captured,flickr_url
0,0.0,0.0,0.0,1.0,0.0,0,"[[453.0, 292.1, 457.0, 253.1, 439.0, 245.1, 43...",21258.0,262145,"[387.0, 71.1, 145.0, 322.0]",1,1218400,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...
1,0.0,0.0,0.0,1.0,0.0,0,"[[505.24, 98.89, 505.4, 96.89, 505.7, 95.04, 5...",74.23865,262145,"[494.31, 91.04, 11.7, 16.16]",1,1220136,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...
2,0.0,0.0,0.0,1.0,0.0,0,"[[282.37, 174.17, 282.91, 161.84, 282.91, 145....",2053.16515,262145,"[282.37, 86.79, 32.16, 97.02]",1,1246711,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...
3,0.0,0.0,0.0,1.0,0.0,0,"[[526.17, 131.72, 530.84, 141.64, 536.68, 142....",9110.87005,262145,"[526.17, 44.14, 90.5, 188.0]",1,1249960,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...
4,0.0,0.0,0.0,1.0,0.0,0,"[[508.34, 166.53, 513.54, 161.32, 516.35, 153....",1962.25295,262145,"[491.51, 81.19, 40.46, 85.34]",1,1262984,person,2,000000262145.jpg,http://images.cocodataset.org/train2017/000000...,427,640,2013-11-20 02:07:55,http://farm8.staticflickr.com/7187/6967031859_...


In [32]:
cleaned_df.shape

(329487, 20)

In [33]:
# Convert dataframe to CSV and save to Kaggle's working directory
cleaned_df.to_csv('/kaggle/working/filtered_coco_metadata.csv', index=False)